# 05 — LoRA Fine-Tuning

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/winstonsmith1897/DantinoX/blob/main/docs/notebooks/05_lora_fine_tuning.ipynb)

**Low-Rank Adaptation (LoRA)** injects small trainable adapter matrices into frozen base model weights.

Key concepts:
- `use_lora=True` in `ModelConfig` injects rank-`r` adapters
- Only adapter weights `A ∈ ℝ^(r×d_in)` and `B ∈ ℝ^(d_out×r)` are trained
- `lora_targets`: `'attention'` · `'ffn'` · `'all'`
- `lora_rank`: adapter capacity (higher = more expressive, more params)
- `lora_alpha`: effective LR scaling — output = `W + (α/r)·B·A`
- `merge_lora()`: fold adapters into base weights for zero-overhead inference

**Runtime**: GPU (T4 or better)

In [ ]:
!pip install -q git+https://github.com/winstonsmith1897/DantinoX.git#egg=dantinox[all]

In [ ]:
import urllib.request, os

if not os.path.exists('tiny_shakespeare.txt'):
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt',
        'tiny_shakespeare.txt')

if not os.path.exists('bible.txt'):
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/mxw/grmr/master/src/finaltests/bible.txt',
        'bible.txt')

print('Data ready.')

## 1. Pre-train a Base Model

Train a small AR model on Tiny Shakespeare. We'll fine-tune it on Biblical text to demonstrate domain adaptation.

In [ ]:
import dantinox as dx

base_run = dx.fit(
    'ar', 'tiny_shakespeare.txt',
    dim=256, n_heads=4, head_size=64, num_blocks=4, vocab_size=200,
    lr=3e-4, epochs=2, batch_size=16,
)
print('Base model checkpoint:', base_run)

## 2. LoRA Configuration

Set `use_lora=True` in `ModelConfig` to activate LoRA. The base model's learned weights remain **frozen**; only the tiny adapter matrices `A` and `B` are trained.

```
W_effective = W_base (frozen) + (α/r) · B @ A
```

At init `B = 0`, so the adapter starts as an identity perturbation.

In [ ]:
import dataclasses, jax
from flax import nnx
from dantinox.core.lora import LoRAParam

def lora_stats(cfg):
    """Build a model and return (total_params, lora_params, lora_pct)."""
    m  = dx.ARParadigm(cfg).build_model(nnx.Rngs(0))
    ap = nnx.state(m, nnx.Param)
    lp = nnx.state(m, LoRAParam)
    total = sum(x.size for x in jax.tree_util.tree_leaves(ap))
    lora  = sum(x.size for x in jax.tree_util.tree_leaves(lp))
    return total, lora, 100 * lora / max(total, 1)

base_cfg = dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                           vocab_size=200, causal=True)

print(f'{"Config":30s}  {"Total":>10s}  {"LoRA":>8s}  {"LoRA%":>6s}')
for rank in (2, 4, 8, 16):
    cfg = dataclasses.replace(base_cfg, use_lora=True,
                               lora_rank=rank, lora_alpha=float(rank*2),
                               lora_targets='attention', lora_dropout=0.0)
    t, l, p = lora_stats(cfg)
    print(f'attention r={rank:2d}                  {t/1e6:>8.2f}M  {l/1e3:>6.1f}K  {p:>6.3f}%')

## 3. LoRA Targets

`lora_targets` controls which weight matrices receive adapters:

| Value | Adapters injected into |
|---|---|
| `'attention'` | Q, K, V, O projections |
| `'ffn'` | FFN up/down/gate projections |
| `'all'` | attention + FFN |

In [ ]:
print(f'{"lora_targets":12s}  {"Total":>10s}  {"LoRA":>8s}  {"LoRA%":>6s}')
for target in ('attention', 'ffn', 'all'):
    cfg = dataclasses.replace(base_cfg, use_lora=True, lora_rank=8, lora_alpha=16.0,
                               lora_targets=target, lora_dropout=0.0)
    t, l, p = lora_stats(cfg)
    print(f'{target:12s}  {t/1e6:>8.2f}M  {l/1e3:>6.1f}K  {p:>6.3f}%')

## 4. Fine-Tuning with LoRA

The `Trainer` automatically detects `use_lora=True` and freezes base weights. Only the LoRA adapters are passed to the optimizer.

In [ ]:
lora_cfg = dx.ModelConfig(
    dim=256, n_heads=4, head_size=64, num_blocks=4, vocab_size=200, causal=True,
    use_lora     = True,
    lora_rank    = 8,
    lora_alpha   = 16.0,
    lora_targets = 'attention',
    lora_dropout = 0.05,
)

ft_run = dx.Trainer(
    dx.ARParadigm(lora_cfg),
    dx.TrainingConfig(lr=1e-3, epochs=2, batch_size=16),
).fit('bible.txt')
print('Fine-tuned checkpoint:', ft_run)

## 5. Compare Base vs Fine-Tuned Generation

In [ ]:
prompt = 'In the beginning'

print('=== Base model (Tiny Shakespeare domain) ===')
print(dx.quick_generate(base_run, prompt, max_new_tokens=150))

print('\n=== LoRA fine-tuned (Bible domain) ===')
print(dx.quick_generate(ft_run, prompt, max_new_tokens=150))

## 6. Merging Adapters

`merge_lora()` folds `(α/r)·B·A` into the base weight `W`, producing a standard model with zero adapter overhead at inference.

In [ ]:
from dantinox.core.lora import merge_lora

# Load the fine-tuned LoRA model
lora_model = dx.load(ft_run, paradigm=dx.ARParadigm(lora_cfg))

# Merge: returns a standard Transformer (no LoRA layers)
merged = merge_lora(lora_model)

total_lora   = sum(x.size for x in jax.tree_util.tree_leaves(nnx.state(lora_model, nnx.Param)))
total_merged = sum(x.size for x in jax.tree_util.tree_leaves(nnx.state(merged, nnx.Param)))
print(f'LoRA model params (Param): {total_lora:,}')
print(f'Merged model params:       {total_merged:,}')
print(f'Overhead removed:          {total_lora - total_merged:,} LoRA-only params')

## 7. Using `dataclasses.replace` for Config Variants

The cleanest way to create LoRA variants of an existing config:

In [ ]:
import dataclasses

base_model_cfg = dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                                 vocab_size=200, causal=True)

# Attention-only LoRA
attn_lora_cfg = dataclasses.replace(
    base_model_cfg,
    use_lora=True, lora_rank=8, lora_alpha=16.0,
    lora_targets='attention', lora_dropout=0.0,
)

# FFN-only LoRA
ffn_lora_cfg = dataclasses.replace(
    base_model_cfg,
    use_lora=True, lora_rank=4, lora_alpha=8.0,
    lora_targets='ffn', lora_dropout=0.0,
)

for name, cfg in [('Attention LoRA', attn_lora_cfg), ('FFN LoRA', ffn_lora_cfg)]:
    rd = dx.Trainer(
        dx.ARParadigm(cfg),
        dx.TrainingConfig(lr=1e-3, epochs=1, batch_size=16),
    ).fit('bible.txt', run_dir=f'/tmp/lora_{name.replace(" ", "_").lower()}')
    print(f'{name}: {dx.quick_generate(rd, prompt, max_new_tokens=60)[:80]}')

## 8. Rank Ablation

Sweep over different LoRA ranks to study the tradeoff between adapter capacity and parameter budget.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

ranks = [2, 4, 8, 16, 32]
lora_counts = []
pcts = []

base_total, _, _ = lora_stats(base_cfg)
for r in ranks:
    cfg = dataclasses.replace(base_cfg, use_lora=True,
                               lora_rank=r, lora_alpha=float(r*2),
                               lora_targets='all', lora_dropout=0.0)
    _, lora_n, lora_p = lora_stats(cfg)
    lora_counts.append(lora_n)
    pcts.append(lora_p)

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(range(len(ranks)), pcts, color='steelblue', alpha=0.8)
ax.set_xticks(range(len(ranks)))
ax.set_xticklabels([f'r={r}' for r in ranks])
ax.set_ylabel('LoRA params (% of base)')
ax.set_title('LoRA parameter budget vs. rank (targets=all)')
for i, (p, n) in enumerate(zip(pcts, lora_counts)):
    ax.text(i, p + 0.01, f'{n/1e3:.0f}K', ha='center', va='bottom', fontsize=8)
plt.tight_layout(); plt.show()